In [65]:
!pip install matplotlib pandas seaborn numpy nltk

In [3]:
%matplotlib inline

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import _pickle as cPickle
import nltk

### Data preparation (same as in other notebook)

In [4]:
df = pd.read_csv('lsapp.tsv', sep='\t')
df_open = df.loc[df['event_type'] == 'Opened']

In [5]:
df_open.loc[:,'timestamp'] = pd.to_datetime(df_open['timestamp'])
df.loc[:,'timestamp'] = pd.to_datetime(df['timestamp'])

In [6]:
df

,user_id,session_id,timestamp,app_name,event_type
0,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Closed
2,0,1,2018-01-16 06:01:07,Minesweeper Classic (Mines),Opened
3,0,1,2018-01-16 06:01:07,Minesweeper Classic (Mines),Closed
4,0,1,2018-01-16 06:01:08,Minesweeper Classic (Mines),Opened
...,...,...,...,...,...
3658584,291,76247,2018-04-06 14:35:15,Facebook,Closed
3658585,291,76247,2018-04-06 14:35:15,Facebook,Opened
3658586,291,76247,2018-04-06 14:35:37,Facebook,Closed
3658587,291,76247,2018-04-06 14:35:37,Facebook,Opened


In [7]:
df['time_dff']  = df[['timestamp']].diff()
df['interaction_id'] = (((df.timestamp-df.timestamp.shift(1) > pd.Timedelta(1, 'm')) 
                              & (df.event_type == 'Opened')) 
                              | ~(df.app_name == df.app_name.shift(1))
                              | ~(df.user_id == df.user_id.shift(1))).cumsum()
df['session_id'] = (((df.timestamp-df.timestamp.shift(1) > pd.Timedelta(5, 'm')) 
                              & (df.event_type == 'Opened')) 
                              | ~(df.user_id == df.user_id.shift(1))).cumsum()
df_start = df.drop_duplicates(subset=['interaction_id'], keep='first')
df_end = df.drop_duplicates(subset=['interaction_id'], keep='last')
df_start.set_index('interaction_id', inplace=True)
df_end.set_index('interaction_id', inplace=True)
df_start.loc[:, 'open_time'] = df_start['timestamp']
df_start.loc[:, 'close_time']  = df_end['timestamp']

C:\Users\Brin\AppData\Local\Temp\ipykernel_22908\4188201099.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_start.loc[:, 'open_time'] = df_start['timestamp']
C:\Users\Brin\AppData\Local\Temp\ipykernel_22908\4188201099.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_start.loc[:, 'close_time']  = df_end['timestamp']


In [8]:
df_start

,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time
interaction_id,,,,,,,,
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09
2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17
3,0,1,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54
4,0,1,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10
5,0,1,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21
...,...,...,...,...,...,...,...,...
599631,291,33218,2018-04-06 12:47:28,Facebook Messenger,Opened,0 days 00:03:29,2018-04-06 12:47:28,2018-04-06 12:53:13
599632,291,33219,2018-04-06 13:20:12,Settings,Opened,0 days 00:26:59,2018-04-06 13:20:12,2018-04-06 13:20:14
599633,291,33220,2018-04-06 14:34:15,Settings,Opened,0 days 01:14:01,2018-04-06 14:34:15,2018-04-06 14:34:17


## App usage records

In [9]:
len(df_start)

599635

## Sessions

In [10]:
len(df_start['session_id'].unique())

33220

## Unique apps

In [11]:
len(df_start['app_name'].unique())

87

## Users

In [12]:
len(df['user_id'].unique())

292

## Mean duration/user 

In [13]:
df_start
grouped = df_start.groupby('user_id')['timestamp'].agg(['min', 'max'])

# Compute duration for each user
grouped['duration'] = grouped['max'] - grouped['min']

# Compute duration for each user in seconds
grouped['duration_seconds'] = grouped['duration'].apply(lambda x: x.total_seconds())

# Compute the mean duration
mean_duration_seconds = grouped['duration_seconds'].mean()

# Convert the mean duration back to a Timedelta object for better readability
mean_duration = pd.to_timedelta(mean_duration_seconds, unit='s')

print(mean_duration)

15 days 16:29:39.106164383


## Mean session time length

### Raw

In [14]:
df_start
grouped = df_start.groupby('session_id')

In [15]:
grouped = df_start.groupby('session_id')['timestamp'].agg(['min', 'max'])

# Compute duration for each session
grouped['duration'] = grouped['max'] - grouped['min']

# Compute duration for each user in seconds
grouped['duration_seconds'] = grouped['duration'].apply(lambda x: x.total_seconds())

# Compute the mean duration
mean_duration_seconds = grouped['duration_seconds'].mean()

# Compute the standard deviation
std_duration_seconds = grouped['duration_seconds'].std()

# Convert the mean duration back to a Timedelta object for better readability
mean_duration = pd.to_timedelta(mean_duration_seconds, unit='s')
std_duration = pd.to_timedelta(std_duration_seconds, unit='s')

print("Mean session time length:", mean_duration)
print("Standard Deviation of session time length:", std_duration)

Mean Duration: 0 days 01:16:28.070288983
Standard Deviation of Duration: 0 days 15:57:34.451853517


### Filtered

In [16]:
threshold = 2

long_sessions = grouped['duration'] > pd.Timedelta(hours=threshold)

# Compute the percentage
percentage_long_sessions = (long_sessions.sum() / len(grouped)) * 100

print(f'Percentage of sessions > {threshold}hr: ', percentage_long_sessions)

grouped = grouped[grouped['duration'] <= pd.Timedelta(hours=threshold)]

# Compute the mean duration
mean_duration_seconds = grouped['duration_seconds'].mean()

# Convert the mean duration back to a Timedelta object for better readability
mean_duration = pd.to_timedelta(mean_duration_seconds, unit='s')

print(mean_duration)

Percentage of sessions > 2hr:  10.045153521974715
0 days 00:17:43.092427132


## Median session time length

In [17]:

grouped = df_start.groupby('session_id')['timestamp'].agg(['min', 'max'])

# Compute duration for each session
grouped['duration'] = grouped['max'] - grouped['min']

# Compute duration for each user in seconds
grouped['duration_seconds'] = grouped['duration'].apply(lambda x: x.total_seconds())

# Compute the median duration
median_duration_seconds = grouped['duration_seconds'].median()

median_duration = pd.to_timedelta(median_duration_seconds, unit='s')

print("Median Duration:", median_duration)

Median Duration: 0 days 00:07:17


## Mean unique apps in each session

In [20]:
unique_apps_per_session = df_start.groupby('session_id')['app_name'].nunique()

mean_unique_apps = unique_apps_per_session.mean()
std_unique_apps = unique_apps_per_session.std()

print("Mean number of unique apps per session:", mean_unique_apps)
print("Standard Deviation of unique apps per session:", std_unique_apps)

Mean number of unique apps per session: 3.4130644190246837
Standard Deviation of unique apps per session: 2.5378790254828716


## Median unique apps in each session

In [21]:
median_unique_apps = unique_apps_per_session.median()
print("Median number of unique apps per session:", median_unique_apps)

Median number of unique apps per session: 3.0


## Mean app switches within a session

In [29]:
df_start['app_switch'] = (df_start['app_name'] != df_start['app_name'].shift()).astype(int)

session_switches = df_start.groupby('session_id')['app_switch'].sum()

mean_switches = session_switches.mean()
std_switches = session_switches.std()

print("Average (mean) of session switches:", mean_switches)
print("Standard deviation of session switches:", std_switches)

Average (mean) of session switches: 16.889283564118003
Standard deviation of session switches: 49.99798689013069


C:\Users\Brin\AppData\Local\Temp\ipykernel_22908\2587565897.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_start['app_switch'] = (df_start['app_name'] != df_start['app_name'].shift()).astype(int)


## Median app switches within a session

In [30]:
median_switches = session_switches.median()

print("Median of session switches:", median_switches)

Median of session switches: 4.0


## Session time

### Session time =  last interaction close time - first interaction open time

In [54]:
# session time as difference between first session usage open_time and last session usage close_time
sessions = df_start.groupby('session_id')

result = df_start.groupby('session_id').agg(
    min_open_time=('open_time', 'min'),
    max_close_time=('close_time', 'max')
)

# save difference as session duration
result['duration'] = result['max_close_time'] - result['min_open_time']

result
# df_session_time = df_start.groupby('session_id').apply(lambda x: x['close_time'].max() - x['open_time'].min())
# df_session_time.head(50)

,min_open_time,max_close_time,duration
session_id,,,
1,2018-01-16 06:01:05,2018-01-16 06:36:26,0 days 00:35:21
2,2018-01-16 07:15:56,2018-01-16 07:21:44,0 days 00:05:48
3,2018-01-16 08:02:05,2018-01-16 08:04:11,0 days 00:02:06
4,2018-01-16 08:30:44,2018-01-16 08:34:14,0 days 00:03:30
5,2018-01-16 09:45:51,2018-01-16 10:29:54,0 days 00:44:03
...,...,...,...
33216,2018-04-06 07:14:47,2018-04-06 07:18:23,0 days 00:03:36
33217,2018-04-06 07:51:54,2018-04-06 12:24:15,0 days 04:32:21
33218,2018-04-06 12:37:57,2018-04-06 12:53:13,0 days 00:15:16


In [57]:
# Compute mean and standard deviation of session times
result['duration'] = pd.to_timedelta(result['duration'])
result['duration_seconds'] = result['duration'].dt.total_seconds()
mean_duration_seconds = result['duration_seconds'].mean()
median_duration_seconds = result['duration_seconds'].median()
std_dev_duration_seconds = result['duration_seconds'].std()

# If you want, convert the results back to Timedelta format
mean_duration = pd.to_timedelta(mean_duration_seconds, unit='s')
median_duration = pd.to_timedelta(median_duration_seconds, unit='s')
std_dev_duration = pd.to_timedelta(std_dev_duration_seconds, unit='s')
                                   
print('Mean duration: ', mean_duration)
print('Median duration: ', median_duration)
print('Standard deviation: +- ', std_dev_duration)

Mean duration:  0 days 01:21:17.673329320
Median duration:  0 days 00:10:20
Standard deviation: +-  0 days 16:01:32.896264631


## Prediction

### Setup

In [25]:
# print number of users by counting unique user ids
df_start['user_id'].nunique()

292

In [21]:
# for each user create its own dataframe
df_user = df_start.groupby('user_id')

In [24]:
# print 10 rows of first user
df_user.get_group(0).head(10)

,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time
interaction_id,,,,,,,,
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09
2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17
3,0,1,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54
4,0,1,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10
5,0,1,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21
6,0,1,2018-01-16 06:26:21,Google,Opened,0 days 00:00:00,2018-01-16 06:26:21,2018-01-16 06:26:26
7,0,1,2018-01-16 06:26:26,Google Chrome,Opened,0 days 00:00:00,2018-01-16 06:26:26,2018-01-16 06:36:26
8,0,2,2018-01-16 07:15:56,Minesweeper Classic (Mines),Opened,0 days 00:39:30,2018-01-16 07:15:56,2018-01-16 07:15:58
9,0,2,2018-01-16 07:20:45,Google Chrome,Opened,0 days 00:04:47,2018-01-16 07:20:45,2018-01-16 07:20:45


In [27]:
# for user 0 print all unique app names
df_user.get_group(0)['app_name'].unique()

array(['Minesweeper Classic (Mines)', 'Gmail', 'Google', 'Instagram',
       'Google Chrome', 'Clock', 'Maps', 'YouTube', 'Facebook',
       'Messages', 'Phone', 'Snapchat', 'Settings', 'Google Photos',
       'Hangouts', 'Amazon Shopping', 'Facebook Messenger',
       'Google Play Store', 'Calendar'], dtype=object)

### Random app

In [38]:
# create user to app dictionary including all unique app names for each user
user_to_app_dict = {}

for user_id, user_df in df_user:
    user_to_app_dict[user_id] = user_df['app_name'].unique()

# print user to app dictionary
user_to_app_dict


{0: array(['Minesweeper Classic (Mines)', 'Gmail', 'Google', 'Instagram',
        'Google Chrome', 'Clock', 'Maps', 'YouTube', 'Facebook',
        'Messages', 'Phone', 'Snapchat', 'Settings', 'Google Photos',
        'Hangouts', 'Amazon Shopping', 'Facebook Messenger',
        'Google Play Store', 'Calendar'], dtype=object),
 1: array(['Google Photos', 'Discord', 'Messages', 'Google', 'Snapchat',
        'Instagram', 'Google Chrome', 'Facebook Messenger', 'Facebook',
        'Google Drive', 'Twitter', 'Google Play Store', 'YouTube', 'Phone',
        'Spotify Music', 'Clock', 'Settings', 'Reddit'], dtype=object),
 2: array(['Android In Call UI', 'Receipt Hog', 'Gmail', 'Facebook',
        'Google Chrome', 'Instagram', 'Amazon Shopping', 'Google',
        'Snapchat', 'Maps', 'Ibotta', 'YouTube', 'Spotify Music',
        'Google Play Store', 'Settings', 'Facebook Messenger'],
       dtype=object),
 3: array(['PayPal Mobile Cash', 'Google Play Store', 'Google Chrome',
        'YouTube', 'A

In [40]:
# iterate through all df_start interactions and choose app_name as random app from user_to_app_dict for user_id
df_start['random_app_name'] = df_start.apply(lambda row: np.random.choice(user_to_app_dict[row['user_id']]), axis=1)
df_start.head(100)

C:\Users\Brin\AppData\Local\Temp\ipykernel_17340\1922601193.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_start['random_app_name'] = df_start.apply(lambda row: np.random.choice(user_to_app_dict[row['user_id']]), axis=1)


,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time,random_app_name
interaction_id,,,,,,,,,
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09,Google Play Store
2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17,Google Play Store
3,0,1,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54,Calendar
4,0,1,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10,Amazon Shopping
5,0,1,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21,Google Chrome
...,...,...,...,...,...,...,...,...,...
96,0,23,2018-01-16 22:32:56,Gmail,Closed,0 days 00:00:01,2018-01-16 22:32:56,2018-01-16 22:32:56,Google
97,0,23,2018-01-16 22:32:57,Phone,Opened,0 days 00:00:01,2018-01-16 22:32:57,2018-01-16 22:34:07,Hangouts
98,0,23,2018-01-16 22:34:07,Gmail,Opened,0 days 00:00:00,2018-01-16 22:34:07,2018-01-16 22:34:47,Amazon Shopping


In [43]:
# create dicitonary user to accuracy
user_to_accuracy_dict = {}

# go through all interactions and check if app_name is in random_app_name for each user and save accuracy
for user_id, user_df in df_user:
    # iterate through all interactions for user_id
    accuracy = 0

    for index, row in user_df.iterrows():
        if row['app_name'] == row['random_app_name']:
            accuracy += 1

    user_to_accuracy_dict[user_id] = accuracy / len(user_df)

# print user to accuracy dictionary
user_to_accuracy_dict

{0: 0.060836501901140684,
 1: 0.06334841628959276,
 2: 0.06521739130434782,
 3: 0.08284023668639054,
 4: 0.13157894736842105,
 5: 0.03174045443897868,
 6: 0.04729082902546429,
 7: 0.043726235741444866,
 8: 0.13333333333333333,
 9: 0.046118721461187215,
 10: 0.125,
 11: 0.06609442060085836,
 12: 0.03162055335968379,
 13: 0.05153203342618384,
 14: 0.1702127659574468,
 15: 0.06201550387596899,
 16: 0.04830917874396135,
 17: 0.10493827160493827,
 18: 0.04808767613509282,
 19: 0.07471264367816093,
 20: 0.10975609756097561,
 21: 0.06164383561643835,
 22: 0.03260869565217391,
 23: 0.07058823529411765,
 24: 0.1717171717171717,
 25: 0.05437597099948213,
 26: 0.06127966356263142,
 27: 0.11194029850746269,
 28: 0.055299539170506916,
 29: 0.0914396887159533,
 30: 0.08053691275167785,
 31: 0.08647575843493054,
 32: 0.14942528735632185,
 33: 0.05409643277146217,
 34: 0.07954545454545454,
 35: 0.14285714285714285,
 36: 0.06561859193438141,
 37: 0.06218487394957983,
 38: 0.06769596199524941,
 39: 0.04

#### Results

In [44]:
# Calculate mean and standard deviation of accuracy
mean_accuracy = np.mean(list(user_to_accuracy_dict.values()))
std_accuracy = np.std(list(user_to_accuracy_dict.values()))

mean_accuracy, std_accuracy

(0.09545466117908077, 0.15162234054750554)

### Most popular app

In [48]:
# create user most popular app dictionary
user_to_most_popular_app_dict = {}

# for each user iterate through all interactions and count app_name occurences
for user_id, user_df in df_user:
    # iterate through all interactions for user_id
    app_name_to_count_dict = {}

    for index, row in user_df.iterrows():
        if row['app_name'] in app_name_to_count_dict:
            app_name_to_count_dict[row['app_name']] += 1
        else:
            app_name_to_count_dict[row['app_name']] = 1

    print('user_id: ', user_id)
    print('app_name_to_count_dict: ', app_name_to_count_dict)
    # find most popular app_name for user_id
    most_popular_app_name = max(app_name_to_count_dict, key=app_name_to_count_dict.get)
    user_to_most_popular_app_dict[user_id] = most_popular_app_name

# print user to most popular app dictionary
user_to_most_popular_app_dict

user_id:  0
app_name_to_count_dict:  {'Minesweeper Classic (Mines)': 174, 'Gmail': 50, 'Google': 170, 'Instagram': 21, 'Google Chrome': 117, 'Clock': 57, 'Maps': 39, 'YouTube': 3, 'Facebook': 20, 'Messages': 70, 'Phone': 35, 'Snapchat': 2, 'Settings': 6, 'Google Photos': 4, 'Hangouts': 1, 'Amazon Shopping': 3, 'Facebook Messenger': 9, 'Google Play Store': 7, 'Calendar': 1}
user_id:  1
app_name_to_count_dict:  {'Google Photos': 4, 'Discord': 80, 'Messages': 15, 'Google': 84, 'Snapchat': 55, 'Instagram': 19, 'Google Chrome': 46, 'Facebook Messenger': 34, 'Facebook': 18, 'Google Drive': 1, 'Twitter': 13, 'Google Play Store': 8, 'YouTube': 6, 'Phone': 9, 'Spotify Music': 35, 'Clock': 2, 'Settings': 2, 'Reddit': 11}
user_id:  2
app_name_to_count_dict:  {'Android In Call UI': 12, 'Receipt Hog': 6, 'Gmail': 62, 'Facebook': 16, 'Google Chrome': 73, 'Instagram': 15, 'Amazon Shopping': 6, 'Google': 6, 'Snapchat': 9, 'Maps': 4, 'Ibotta': 2, 'YouTube': 1, 'Spotify Music': 2, 'Google Play Store': 5

{0: 'Minesweeper Classic (Mines)',
 1: 'Google',
 2: 'Google Chrome',
 3: 'Google Chrome',
 4: 'Google Chrome',
 5: 'Twitter',
 6: 'Reddit',
 7: 'Google',
 8: 'Words With Friends 2',
 9: 'Google Chrome',
 10: 'Google Chrome',
 11: 'Samsung Internet Browser',
 12: 'WhatsApp Messenger',
 13: 'Facebook',
 14: 'Google Chrome',
 15: 'Gmail',
 16: 'Slidejoy',
 17: 'Discord',
 18: 'Google Chrome',
 19: 'Messages',
 20: 'Google Chrome',
 21: 'Samsung Email',
 22: 'Facebook',
 23: 'Samsung Email',
 24: 'Google',
 25: 'Google Chrome',
 26: 'Android In Call UI',
 27: 'Google Chrome',
 28: 'Google Chrome',
 29: 'Google Chrome',
 30: 'Contacts',
 31: 'Telegram',
 32: 'Google',
 33: 'Samsung Internet Browser',
 34: 'Google Chrome',
 35: 'Google Chrome',
 36: 'YouTube',
 37: 'Gmail',
 38: 'Yahoo Mail',
 39: 'Google',
 40: 'Facebook Messenger',
 41: 'Facebook',
 42: 'Google',
 43: 'Google Chrome',
 44: 'Google Chrome',
 45: 'Google Play Store',
 46: 'Contacts',
 47: 'Google Chrome',
 48: 'Twitter',
 4

In [49]:
# iterate through all df_start interactions and choose app_name as most popular app for user_id
df_start['most_popular_app_name'] = df_start.apply(lambda row: user_to_most_popular_app_dict[row['user_id']], axis=1)

# create dicitonary user to accuracy
user_to_accuracy_dict = {}

# go through all interactions and check if app_name is in random_app_name for each user and save accuracy
for user_id, user_df in df_user:
    # iterate through all interactions for user_id
    accuracy = 0

    for index, row in user_df.iterrows():
        if row['app_name'] == row['most_popular_app_name']:
            accuracy += 1

    user_to_accuracy_dict[user_id] = accuracy / len(user_df)

# print user to accuracy dictionary
user_to_accuracy_dict

C:\Users\Brin\AppData\Local\Temp\ipykernel_17340\596777393.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_start['most_popular_app_name'] = df_start.apply(lambda row: user_to_most_popular_app_dict[row['user_id']], axis=1)


{0: 0.22053231939163498,
 1: 0.19004524886877827,
 2: 0.3173913043478261,
 3: 0.40236686390532544,
 4: 0.26973684210526316,
 5: 0.3367299133286484,
 6: 0.24717595251771013,
 7: 0.23479087452471484,
 8: 0.36666666666666664,
 9: 0.31689497716894977,
 10: 0.42105263157894735,
 11: 0.3344778254649499,
 12: 0.15019762845849802,
 13: 0.24373259052924792,
 14: 0.48936170212765956,
 15: 0.18733850129198967,
 16: 0.32971014492753625,
 17: 0.29012345679012347,
 18: 0.2516215611719973,
 19: 0.15517241379310345,
 20: 0.4146341463414634,
 21: 0.23835616438356164,
 22: 0.17527173913043478,
 23: 0.19411764705882353,
 24: 0.42424242424242425,
 25: 0.2553081305023304,
 26: 0.2547311504956443,
 27: 0.4253731343283582,
 28: 0.2073732718894009,
 29: 0.39299610894941633,
 30: 0.21476510067114093,
 31: 0.3966543804933371,
 32: 0.5402298850574713,
 33: 0.18659349274794199,
 34: 0.20454545454545456,
 35: 0.2857142857142857,
 36: 0.23513328776486672,
 37: 0.19831932773109243,
 38: 0.1828978622327791,
 39: 0.36

In [50]:
# Calculate mean and standard deviation of accuracy
mean_accuracy = np.mean(list(user_to_accuracy_dict.values()))
std_accuracy = np.std(list(user_to_accuracy_dict.values()))

mean_accuracy, std_accuracy

(0.3247922031056106, 0.15336049330744814)

### Most recently used